# The CLIP loss in NumPy

A photo-sharing site trains a CLIP-style model on photos and the captions people wrote for
them. A photo often has several captions, so a batch drawn by caption can hold the same
photo twice, each time with its own random crop. You will write the symmetric loss so that
it gives the right answer at any temperature, extend it to batches in which a photo appears
more than once, and work out its gradient with respect to the logits.

**How to work through it.** Each part asks for one function. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your function is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first.

In [ ]:
import numpy as np

# A batch of 64 captions drawn from 40 photos, so some photos come up more than once.
# photo_of[i] is the photo behind pair i. Each appearance of a photo is a fresh random
# crop, so two rows of the same photo have close but different image embeddings.
rng = np.random.default_rng(5)
n_photos, d = 40, 16
photo_vec = rng.normal(size=(n_photos, d))
photo_of = rng.integers(0, n_photos, 64)
img_batch = photo_vec[photo_of] + 0.15 * rng.normal(size=(64, d))
txt_batch = photo_vec[photo_of] + 0.3 * rng.normal(size=(64, d))
print(f"64 pairs from {len(np.unique(photo_of))} different photos")

## Part 1 (coding task): The symmetric loss at any temperature

Write `clip_loss(img, txt, tau)`. Row $i$ of `img` and row $i$ of `txt` are a matching
pair. Scale every row of both arrays to length 1, form the logits
$Z_{ij} = \cos(\text{img}_i, \text{txt}_j)/\tau$, and return `(loss, i2t, t2i)`:

- `i2t`, the image-to-text loss: the mean over rows $i$ of the cross-entropy of row $i$ of
  $Z$ with target $i$,
- `t2i`, the text-to-image loss: the same over the columns of $Z$,
- `loss`, their average.

The check runs it at temperatures down to $\tau = 0.001$.

In [ ]:
def clip_loss(img, txt, tau):
    """Symmetric CLIP loss for matching rows. Returns (loss, i2t, t2i)."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    # TODO
    raise NotImplementedError

In [ ]:
img_a = np.array([[2.0, 0, 0], [0, 1, 0], [1, 1, 0]])
txt_a = np.array([[1.0, 0, 0], [0, 3, 0], [0, 0, 1]])
loss, i2t, t2i = clip_loss(img_a, txt_a, 0.5)
rescaled = clip_loss(img_a * np.array([[5.0], [0.2], [3.0]]), txt_a * np.array([[0.5], [2.0], [7.0]]), 0.5)[0]
assert np.isclose(rescaled, loss), "the loss changed when the rows were rescaled: compare cosine similarities, so scale each row to length 1 first"
assert np.isclose(i2t, 0.900390, atol=1e-6), f"image-to-text loss {i2t:.6f}, expected 0.900390: average the cross-entropies of the rows of Z, one per image"
assert np.isclose(t2i, 0.716813, atol=1e-6), f"text-to-image loss {t2i:.6f}, expected 0.716813: average the cross-entropies of the columns of Z, one per caption"
assert np.isclose(loss, (i2t + t2i) / 2), "loss should be the average of i2t and t2i"
img_b = np.array([[1.0, 0], [0, 1], [1, 1]])
txt_b = np.array([[0.0, 1], [1, 1], [1, 0]])
with np.errstate(all="ignore"):
    for tau, expected in [(0.05, 10.575711), (0.001, 528.595479)]:
        got = clip_loss(img_b, txt_b, tau)[0]
        assert np.isfinite(got), f"at tau = {tau} the loss is {got}: some logits reach {1 / tau:.0f}, and e to that power overflows a float"
        assert np.isclose(got, expected, rtol=1e-6), f"at tau = {tau} the loss is {got:.6f}, expected {expected}"
print("Part 1 passed")

## Part 2 (coding task): A photo that appears twice

Write `clip_loss_shared(img, txt, photo_of, tau)`, where `photo_of[i]` names the photo
behind pair $i$. When several rows share a photo, each of their images has several correct
captions and each of their captions several correct images, so the target is no longer the
diagonal. Give row $i$ a target $y_{i\cdot}$ split evenly over the captions $j$ with
`photo_of[j] == photo_of[i]`, give column $j$ a target split evenly over the images $i$ with
the same photo as $j$, and score each row and each column with the cross-entropy against
its target, $-\sum_j y_{ij} \ln p_{ij}$, where $p$ is that row's or column's softmax.
Return `(loss, i2t, t2i)` as in Part 1. When every photo is different it must agree with
Part 1.

In [ ]:
def clip_loss_shared(img, txt, photo_of, tau):
    """CLIP loss with targets split over every pair of the same photo. Returns (loss, i2t, t2i)."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    photo_of = np.asarray(photo_of)
    # TODO
    raise NotImplementedError

In [ ]:
photo_s = np.array([0, 1, 0, 2, 1])
img_s = np.array([[2.0, 1, 0], [0, 1, 2], [2, 1, 1], [1, 0, 1], [0, 2, 3]])   # rows 0, 2 and rows 1, 4 are two crops of one photo
txt_s = np.array([[2.0, 0, 0], [0, 1, 1], [1, 1, 0], [1, 0, 2], [0, 0, 1]])
loss, i2t, t2i = clip_loss_shared(img_s, txt_s, photo_s, 0.5)
assert np.allclose(clip_loss_shared(img_a, txt_a, [7, 8, 9], 0.5), clip_loss(img_a, txt_a, 0.5)), "with every photo different, the result should equal Part 1's"
assert np.isclose(i2t, 1.169491, atol=1e-6), f"image-to-text loss {i2t:.6f}, expected 1.169491: row i's target puts 1/k on each of the k captions of its photo, and the row's cross-entropy sums target times log-probability"
assert np.isclose(t2i, 1.166061, atol=1e-6), f"text-to-image loss {t2i:.6f}, expected 1.166061: column j's target puts 1/k on each of the k images of its photo"
assert np.isclose(loss, (i2t + t2i) / 2), "loss should be the average of i2t and t2i"
three = clip_loss_shared([[0.0, 0, 1]] * 3, [[1.0, 0, 1], [0, 1, 1], [-1, 0, 1]], [4, 4, 4], 1.0)[0]
assert np.isclose(three, np.log(3)), f"one photo three times, with captions equally close to it, gives {three:.6f}: every softmax is uniform and every target is split three ways, so the loss is ln 3"
print("Part 2 passed")

## Part 3 (coding task): The gradient with respect to the logits

Write `logit_grad(img, txt, photo_of, tau)`, returning the $N \times N$ array whose entry
$(i, j)$ is $\partial \mathcal{L} / \partial Z_{ij}$ for the `loss` of Part 2, where $Z$ is the
logit matrix. One fact to build on: for a single cross-entropy $-\sum_j y_j \ln p_j$ with
$p = \operatorname{softmax}(z)$ and $\sum_j y_j = 1$, the gradient with respect to $z$ is $p - y$.

The check compares your array with how Part 2's loss actually changes when the embeddings
move, then shows what the targets do to the pairs that share a photo.

In [ ]:
def logit_grad(img, txt, photo_of, tau):
    """d loss / d Z for the loss of Part 2, as an N x N array."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    photo_of = np.asarray(photo_of)
    # TODO
    raise NotImplementedError

In [ ]:
G_a = logit_grad(img_a, txt_a, [0, 1, 2], 0.5)
want = np.array([[-0.103666, 0.031082, 0.073307], [0.031082, -0.103666, 0.073307], [0.129135, 0.129135, -0.259714]])
assert np.shape(G_a) == (3, 3), f"for 3 pairs the gradient should be a 3 x 3 array, not shape {np.shape(G_a)}"
assert np.allclose(G_a, want, atol=1e-6), f"on the Part 1 example the gradient is\n{np.round(G_a, 6)}\nexpected\n{want}"


def logits(img, txt, tau):
    U = img / np.linalg.norm(img, axis=1, keepdims=True)
    V = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    return U @ V.T / tau


rng_check = np.random.default_rng(11)
img_c, txt_c = rng_check.normal(size=(5, 6)), rng_check.normal(size=(5, 6))
photo_c = np.array([3, 1, 3, 0, 1])
G = logit_grad(img_c, txt_c, photo_c, 0.3)
for trial in range(3):
    Di, Dt = rng_check.normal(size=(5, 6)), rng_check.normal(size=(5, 6))
    h = 1e-6
    change = (clip_loss_shared(img_c + h * Di, txt_c + h * Dt, photo_c, 0.3)[0]
              - clip_loss_shared(img_c - h * Di, txt_c - h * Dt, photo_c, 0.3)[0]) / (2 * h)
    dZ = (logits(img_c + h * Di, txt_c + h * Dt, 0.3) - logits(img_c - h * Di, txt_c - h * Dt, 0.3)) / (2 * h)
    assert np.isclose(np.sum(G * dZ), change, rtol=1e-5, atol=1e-8), f"moving the embeddings changes the loss at rate {change:.6f}, but your gradient predicts {np.sum(G * dZ):.6f}"

tau = 0.07
same = (photo_of[:, None] == photo_of[None, :]) & ~np.eye(64, dtype=bool)
g_diag = logit_grad(img_batch, txt_batch, np.arange(64), tau)[same].mean()
g_shared = logit_grad(img_batch, txt_batch, photo_of, tau)[same].mean()
print(f"{same.sum()} entries pair an image with another caption of the same photo")
print(f"mean gradient on them, diagonal targets: {g_diag:+.6f}   (descent lowers these logits)")
print(f"mean gradient on them, shared targets:   {g_shared:+.6f}")
assert g_diag > 0 and abs(g_shared) < g_diag / 20, "on the batch, diagonal targets should push these matches apart, and shared targets should leave them nearly alone"
print("Part 3 passed")

## Solution

### Part 1: The symmetric loss at any temperature

In [ ]:
def log_probs(Z, axis):
    """Log of the softmax along one axis, computed without overflow."""
    top = Z.max(axis=axis, keepdims=True)            # after subtracting it the largest entry is 0, so exp cannot overflow
    return Z - top - np.log(np.exp(Z - top).sum(axis=axis, keepdims=True))


def clip_loss(img, txt, tau):
    """Symmetric CLIP loss for matching rows. Returns (loss, i2t, t2i)."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    U = img / np.linalg.norm(img, axis=1, keepdims=True)
    V = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    Z = U @ V.T / tau
    i2t = -np.mean(np.diag(log_probs(Z, axis=1)))  # each image picks its caption along its row
    t2i = -np.mean(np.diag(log_probs(Z, axis=0)))  # each caption picks its image down its column
    return (i2t + t2i) / 2, i2t, t2i

**Why it works.** Subtracting the largest entry of a row before exponentiating leaves the softmax
unchanged, since the factor $e^{-\max}$ cancels between the top and the bottom, and makes the
largest exponential exactly 1. Without it, $e^{1000}$ is `inf` and the loss is `nan`.
The column direction needs its own maximum, taken down each column: a row's maximum can
sit far below the largest entry of a column, and then a whole column underflows to 0.
The two directions use the same matrix with a different axis, which is why they give
different numbers on the same batch.

### Part 2: A photo that appears twice

In [ ]:
def shared_targets(photo_of):
    same = (photo_of[:, None] == photo_of[None, :]).astype(float)
    return same / same.sum(axis=1, keepdims=True)    # each row sums to 1, and so does each column


def clip_loss_shared(img, txt, photo_of, tau):
    """CLIP loss with targets split over every pair of the same photo. Returns (loss, i2t, t2i)."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    photo_of = np.asarray(photo_of)
    U = img / np.linalg.norm(img, axis=1, keepdims=True)
    V = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    Z = U @ V.T / tau
    Y = shared_targets(photo_of)
    n = len(Z)
    i2t = -np.sum(Y * log_probs(Z, axis=1)) / n
    t2i = -np.sum(Y * log_probs(Z, axis=0)) / n
    return (i2t + t2i) / 2, i2t, t2i

**Why it works.** The row target is the 0/1 matrix of "same photo" divided by its row sums, so each row is a
probability distribution, and with all photos distinct it is the identity. Since rows and
columns of the same photo have the same count, the same matrix also serves as the column
targets. A target split $k$ ways can never be matched exactly by a softmax that also gives
weight to other captions, so the loss can go no lower than the entropy of the target,
$\ln k$ for those rows, as the three-copy case shows. If the copies of a photo had
identical embeddings, their rows of $Z$ would be equal and this loss would equal Part 1's
exactly: the split only changes anything when the copies differ, as crops do.

### Part 3: The gradient with respect to the logits

In [ ]:
def logit_grad(img, txt, photo_of, tau):
    """d loss / d Z for the loss of Part 2, as an N x N array."""
    img = np.asarray(img, dtype=float)
    txt = np.asarray(txt, dtype=float)
    photo_of = np.asarray(photo_of)
    U = img / np.linalg.norm(img, axis=1, keepdims=True)
    V = txt / np.linalg.norm(txt, axis=1, keepdims=True)
    Z = U @ V.T / tau
    Y = shared_targets(photo_of)
    R = np.exp(log_probs(Z, axis=1))               # each row's softmax
    C = np.exp(log_probs(Z, axis=0))               # each column's softmax
    n = len(Z)
    return ((R - Y) / n + (C - Y) / n) / 2           # every logit is in one row term and one column term

**Why it works.** Each logit $Z_{ij}$ appears in two cross-entropies, row $i$'s and column $j$'s, so it
collects $(R_{ij} - Y_{ij})/N$ from the image-to-text term and $(C_{ij} - Y_{ij})/N$ from
the text-to-image term, where $R$ and $C$ are the row and column softmaxes, and `loss`
halves their sum. The $1/N$ comes from averaging over rows and columns. Leaving out the
column term gives an array that still sums to 0 along every row, but it describes only half
of the loss, and the check's moving embeddings catch it. On the batch, the diagonal targets
give every "same photo, other caption" logit a positive gradient, so descent pushes each
crop of a photo away from the captions that came with its other crops. With the shared
targets these logits get almost no gradient: the model already gives them about their
share of the target, so there is nothing left to push against.